# Rain Prediction — Spatial design tuning + fresh hyperparameter search (EXP-007)

EXP-006 showed spatial neighbour features broke the same-station ceiling, driven by **same-day**
neighbour humidity/pressure. Two levers remain:

1. **Spatial design** — sweep `K` (1/3/5/7) and try **directional upstream (westerly)** neighbours
   instead of omnidirectional nearest (mid-latitude weather advects W->E).
2. **Fresh hyperparameter search** on the full 44-feature set — the XGBoost config has been frozen
   since EXP-002, *before any spatial features existed*.

Même harnais deux-splits que EXP-005/006, branché sur le pipeline : features de base et split
chronologique depuis `weatherAUS_{X,y}_{train,test}.csv`, features retardées et voisines depuis
`weatherAUS_trend_grouped.csv` (pré-imputation). Le split chronologique est la référence (les
features spatiales sont une aide à l'interpolation qui ne transfère pas aux stations inconnues —
EXP-006). Logs -> `SPTUNE_result.txt`, figures `sptune_*`, modèles `joblib` dans `../../models/`.


## 1. Setup, imports, constants

In [ ]:
# --- Environnement ---
# Ce notebook tourne en local sur les sorties du pipeline de preprocessing
# (dossier ../../data). Sur Colab, installer les dépendances puis pointer
# WEATHER_DATA_DIR vers le dossier contenant les CSV du pipeline.
import subprocess, sys as _sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([_sys.executable, '-m', 'pip', '-q', 'install', '-U', 'xgboost'],
                   check=False)
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], check=False)

print(f'Running on Colab: {IN_COLAB}')


In [ ]:
import os, sys, zipfile, warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    TimeSeriesSplit, GroupKFold, GroupShuffleSplit, RandomizedSearchCV)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    roc_curve, precision_recall_curve, auc)
import xgboost as xgb
from xgboost import XGBClassifier
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')

def _gpu_available():
    try:
        d = xgb.DMatrix(np.zeros((8, 2)), label=np.zeros(8))
        xgb.train({'device': 'cuda', 'tree_method': 'hist'}, d, num_boost_round=1)
        return True
    except Exception as e:
        print(f'CUDA probe failed ({type(e).__name__}); CPU fallback.'); return False

USE_GPU = _gpu_available()
DEVICE = 'cuda' if USE_GPU else 'cpu'
SEARCH_N_JOBS = 1 if USE_GPU else -1
XGB_KW = dict(tree_method='hist', device=DEVICE, eval_metric='logloss',
              random_state=RANDOM_STATE, n_jobs=-1)
def make_xgb(**ov):
    p = dict(XGB_KW); p.update(ov); return XGBClassifier(**p)

# Frozen EXP-002 config (the K-sweep baseline; the fresh search will try to beat it).
V2_PARAMS = dict(max_depth=6, min_child_weight=10, gamma=2.0, reg_lambda=20, reg_alpha=0.5,
                 learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_estimators=800)
print(f'xgboost {xgb.__version__} | device: {DEVICE}')

### Output setup

In [ ]:
os.makedirs('figures', exist_ok=True)
_PATH = 'SPTUNE_result.txt'
_FILE = open(_PATH, 'w', encoding='utf-8')
class _Tee:
    def __init__(self, *s): self._s = s
    def write(self, d):
        for s in self._s: s.write(d)
    def flush(self):
        for s in self._s: s.flush()
    def isatty(self): return False
_ORIG = sys.stdout; sys.stdout = _Tee(_ORIG, _FILE)
print('# Spatial tuning + fresh search — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
def log_section(t):
    print('\n' + '=' * 72 + '\n' + t + '\n' + '=' * 72)
def save_fig(name, fig=None):
    fig = fig or plt.gcf(); p = f'figures/{name}.png'
    fig.savefig(p, dpi=150, bbox_inches='tight'); print(f'  [saved {p}]')

# --- Sérialisation des modèles ---------------------------------------------
# Chaque étape du notebook est enregistrée dans <repo>/models sous forme d'un
# dict joblib : le modèle + les métadonnées nécessaires pour le rejouer.
MODEL_DIR = os.environ.get('WEATHER_MODEL_DIR', '../../models')
os.makedirs(MODEL_DIR, exist_ok=True)
NOTEBOOK = 'EXP-007_XGBoost_spatial_tuned'

def save_model(obj, name, **meta):
    """Sauvegarde `obj` dans MODEL_DIR/<name>.joblib avec ses métadonnées."""
    path = f'{MODEL_DIR}/{name}.joblib'
    joblib.dump({'model': obj,
                 'notebook': NOTEBOOK,
                 'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                 **meta}, path)
    print(f'  [saved {path} ({os.path.getsize(path) / 1e6:.1f} MB)]')


## 2. Load + base + temporal features

In [ ]:
log_section('2. Load + base + temporal features')

# Sorties du pipeline, dans <repo>/data :
#   3.preprocessing_trend_group.ipynb -> weatherAUS_trend_grouped.csv
#   4_preprocessing_imputation.ipynb  -> weatherAUS_MICE_cyclic.csv
#                                        weatherAUS_X_train.csv / weatherAUS_y_train.csv
#                                        weatherAUS_X_test.csv  / weatherAUS_y_test.csv
DATA_DIR = os.environ.get('WEATHER_DATA_DIR', '../../data')

# Matrices déjà splitées, encodées (LabelEncoder) et scalées (RobustScaler ajusté
# sur le train uniquement) par 4_preprocessing_imputation.ipynb.
X_train = pd.read_csv(f'{DATA_DIR}/weatherAUS_X_train.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/weatherAUS_X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/weatherAUS_y_train.csv').squeeze('columns').astype(int)
y_test  = pd.read_csv(f'{DATA_DIR}/weatherAUS_y_test.csv').squeeze('columns').astype(int)

# Index (Date, Location) aligné ligne à ligne sur X_train / X_test :
# weatherAUS_MICE_cyclic.csv est trié par Date et le pipeline y applique le même
# seuil TEST_CUTOFF, donc ses len(X_train) premières lignes SONT celles de X_train.
meta = pd.read_csv(f'{DATA_DIR}/weatherAUS_MICE_cyclic.csv',
                   usecols=['Date', 'Location', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm'], parse_dates=['Date'])
meta = meta[['Date', 'Location', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm']]
assert len(meta) == len(X_train) + len(X_test), 'X_train/X_test ne recouvrent pas MICE_cyclic'
meta_train = meta.iloc[:len(X_train)].reset_index(drop=True)
meta_test  = meta.iloc[len(X_train):].reset_index(drop=True)
assert (meta_train['Date'] < TEST_CUTOFF).all(), 'lignes de train postérieures au seuil'
assert (meta_test['Date'] >= TEST_CUTOFF).all(), 'lignes de test antérieures au seuil'

print(f'X_train {X_train.shape} | X_test {X_test.shape}')
print(f'Train {meta_train["Date"].min().date()} -> {meta_train["Date"].max().date()} | '
      f'Test {meta_test["Date"].min().date()} -> {meta_test["Date"].max().date()}')
print(f'Stations : {meta["Location"].nunique()} | pluie train {y_train.mean():.4%} / test {y_test.mean():.4%}')

# Frame AVANT imputation (mêmes lignes, NaN d'origine conservés) : sert à
# reconstruire les features retardées / voisines sans réutiliser de valeur
# reconstruite par MICE.
pre = pd.read_csv(f'{DATA_DIR}/weatherAUS_trend_grouped.csv', parse_dates=['Date'])
assert len(pre) == len(meta), 'weatherAUS_trend_grouped.csv désaligné du pipeline'
print(f'Pré-imputation : {pre.shape[0]:,} lignes x {pre.shape[1]} colonnes '
      f'({pre.isna().sum().sum():,} NaN conservés)')

# --- Features de base (v2) ---
# Features de tendance intra-journalière. Elles sont calculées sur les valeurs
# NON scalées de weatherAUS_MICE_cyclic.csv : une différence de colonnes déjà
# passées au RobustScaler (échelles distinctes) ne serait pas la même grandeur.
for X, m in ((X_train, meta_train), (X_test, meta_test)):
    X['Pressure_diff'] = (m['Pressure3pm'] - m['Pressure9am']).to_numpy()
    X['Temp_diff']     = (m['Temp3pm']     - m['Temp9am']).to_numpy()

# Le pipeline livre 29 colonnes. On retire les mêmes que l'étude LR pour que les
# expériences EXP-001..007 restent comparables entre elles :
DROP_COLS = [
    'WindGustDir', 'WindDir9am', 'WindDir3pm',  # labels encodés ; les sin/cos sont conservés
    'RainToday',                                # quasi-fuite (on donne déjà le reste de la journée)
    'Humidity9am',                              # on garde Humidity3pm (plus prédictive)
    'Month',                                    # redondant avec Date_sin / Date_cos
]
BASE_FEATURES = [c for c in X_train.columns if c not in DROP_COLS]

# Matrice complète dans l'ordre chronologique du pipeline : X_train puis X_test.
# Les blocs de features additionnels (temporel / spatial) viendront s'y coller.
X_all = pd.concat([X_train[BASE_FEATURES], X_test[BASE_FEATURES]], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
loc_all = meta['Location']
dates   = meta['Date']
N_TRAIN = len(X_train)

def align_to_pipeline(frame, cols):
    """Réaligne un bloc de features indexé par (Date, Location) sur l'ordre des
    lignes du pipeline. Les deux frames partagent la même clé unique, donc le
    réalignement est exact et ne peut pas dupliquer de ligne."""
    key = pd.MultiIndex.from_frame(frame[['Date', 'Location']])
    assert key.is_unique, 'clé (Date, Location) non unique dans le bloc de features'
    out = frame.set_axis(key)[cols].reindex(pd.MultiIndex.from_frame(meta[['Date', 'Location']]))
    return out.reset_index(drop=True)

print(f'BASE features: {len(BASE_FEATURES)} | X_all {X_all.shape}')

# --- Features temporelles (frame pré-imputation) ---
# Toutes les features ci-dessous n'utilisent que les jours <= t-1 (pas d'info du jour
# même, pas de futur), construites par `Location` sur weatherAUS_trend_grouped.csv,
# c'est-à-dire AVANT l'imputation MICE : aucune valeur reconstruite ne peut remonter
# dans un lag. Les NaN qui en résultent (premiers jours / manquants d'origine) sont
# laissés tels quels — XGBoost les gère nativement et EXP-003 a montré que les
# imputer n'apporte rien.
pre_s = pre.sort_values(['Location', 'Date']).copy()
g = pre_s.groupby('Location', sort=False)

LAG_COLS = ['Humidity3pm','Pressure3pm','Rainfall','Cloud3pm','Sunshine','Temp3pm','WindGustSpeed']
feat = pd.DataFrame({'Date': pre_s['Date'].values, 'Location': pre_s['Location'].values})
for c in LAG_COLS:
    feat[f'{c}_lag1'] = g[c].shift(1).values

# multi-day, all strictly past (>= t-1)
feat['Pressure_3d_chg']  = (g['Pressure3pm'].shift(1) - g['Pressure3pm'].shift(4)).values
feat['Humidity_3d_mean'] = g['Humidity3pm'].apply(lambda s: s.shift(1).rolling(3).mean()).values
feat['Rainfall_3d_sum']  = g['Rainfall'].apply(lambda s: s.shift(1).rolling(3).sum()).values

# yesterday's rain flag + days since last rain (computed on the lagged flag -> strictly past)
rt = pre_s['RainToday'].astype(int)          # booléen dans le frame du pipeline
rt_lag = rt.groupby(pre_s['Location'], sort=False).shift(1)
feat['RainToday_lag1'] = rt_lag.values
def _days_since(s):
    out = np.empty(len(s)); cnt = 9999.0
    for i, v in enumerate(s.to_numpy()):
        out[i] = cnt
        cnt = 0.0 if v == 1 else (cnt + 1 if cnt < 9999 else 9999)
    return out
feat['days_since_rain'] = (
    rt_lag.fillna(0).groupby(pre_s['Location'], sort=False).transform(lambda s: _days_since(s)).values)

TEMPORAL_FEATURES = [c for c in feat.columns if c not in ('Date', 'Location')]
X_all[TEMPORAL_FEATURES] = align_to_pipeline(feat, TEMPORAL_FEATURES)
nan_share = X_all[TEMPORAL_FEATURES].isna().mean().mean()
print(f'Temporal features: {len(TEMPORAL_FEATURES)} -> {TEMPORAL_FEATURES}')
print(f'Mean NaN share across temporal features: {nan_share:.1%} (left for XGBoost native handling)')


## 3. Geocoding + neighbour graphs (K=1,3,5,7 nearest, and K=3 upstream-west)

In [ ]:
log_section('3. Neighbour graphs')
COORDS = {
    'Adelaide': (-34.9285, 138.6007), 'Albany': (-35.0269, 117.8837), 'Albury': (-36.0737, 146.9135),
    'AliceSprings': (-23.6980, 133.8807), 'BadgerysCreek': (-33.8829, 150.7609), 'Ballarat': (-37.5622, 143.8503),
    'Bendigo': (-36.7570, 144.2794), 'Brisbane': (-27.4698, 153.0251), 'Cairns': (-16.9186, 145.7781),
    'Canberra': (-35.2809, 149.1300), 'Cobar': (-31.4958, 145.8389), 'CoffsHarbour': (-30.2986, 153.1094),
    'Dartmoor': (-37.9225, 141.2767), 'Darwin': (-12.4634, 130.8456), 'GoldCoast': (-28.0167, 153.4000),
    'Hobart': (-42.8821, 147.3272), 'Katherine': (-14.4652, 132.2635), 'Launceston': (-41.4332, 147.1441),
    'Melbourne': (-37.8136, 144.9631), 'MelbourneAirport': (-37.6690, 144.8410), 'Mildura': (-34.1855, 142.1625),
    'Moree': (-29.4639, 149.8407), 'MountGambier': (-37.8284, 140.7820), 'MountGinini': (-35.5294, 148.7723),
    'Newcastle': (-32.9283, 151.7817), 'Nhil': (-36.3333, 141.6500), 'NorahHead': (-33.2833, 151.5667),
    'NorfolkIsland': (-29.0408, 167.9547), 'Nuriootpa': (-34.4700, 138.9960), 'PearceRAAF': (-31.6676, 116.0292),
    'Penrith': (-33.7507, 150.6877), 'Perth': (-31.9505, 115.8605), 'PerthAirport': (-31.9385, 115.9672),
    'Portland': (-38.3609, 141.6041), 'Richmond': (-33.6000, 150.7667), 'Sale': (-38.1026, 147.0730),
    'SalmonGums': (-32.9815, 121.6438), 'Sydney': (-33.8688, 151.2093), 'SydneyAirport': (-33.9399, 151.1753),
    'Townsville': (-19.2590, 146.8169), 'Tuggeranong': (-35.4244, 149.0888), 'Uluru': (-25.3444, 131.0369),
    'WaggaWagga': (-35.1082, 147.3598), 'Walpole': (-34.9777, 116.7338), 'Watsonia': (-37.7100, 145.0830),
    'Williamtown': (-32.8150, 151.8428), 'Witchcliffe': (-34.0261, 115.1003), 'Wollongong': (-34.4278, 150.8931),
    'Woomera': (-31.1656, 136.8193),
}
assert set(meta['Location'].unique()) <= set(COORDS)
stations = sorted(COORDS)
def haversine(a, b):
    R = 6371.0; p1, p2 = np.radians(a[0]), np.radians(b[0])
    dphi = np.radians(b[0]-a[0]); dl = np.radians(b[1]-a[1])
    h = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(h))

def knn(K, west_only=False):
    nb, nd = {}, {}
    for s in stations:
        cand = sorted((haversine(COORDS[s], COORDS[t]), t) for t in stations if t != s)
        nd[s] = cand[0][0]
        if west_only:
            west = [t for _, t in cand if COORDS[t][1] < COORDS[s][1]]
            rest = [t for _, t in cand if COORDS[t][1] >= COORDS[s][1]]
            nb[s] = (west + rest)[:K]
        else:
            nb[s] = [t for _, t in cand][:K]
    return nb, nd

GRAPHS = {f'K{K}': knn(K)[0] for K in (1, 3, 5, 7)}
GRAPHS['K3_west'] = knn(3, west_only=True)[0]
NEAREST_DIST = knn(1)[1]
print('graphs:', list(GRAPHS))
print('  Sydney  K3      :', GRAPHS['K3']['Sydney'])
print('  Sydney  K3_west :', GRAPHS['K3_west']['Sydney'])

## 4. Spatial feature builder (parameterized by neighbour graph)

Pivots are precomputed once on le frame pré-imputation ; each graph just re-aggregates.
`mean` saute les NaN des voisins, same-day `_t` + lagged `_t1`, plus static `nearest_dist`.
`build_spatial` renvoie un bloc **déjà réaligné sur l'ordre des lignes du pipeline**, prêt à être
collé à `X_all`.


In [ ]:
log_section('4. Spatial feature builder')

pre2 = pre.assign(_RainToday=pre['RainToday'].astype(int))
full_dates = pd.date_range(pre['Date'].min(), pre['Date'].max(), freq='D')
WIDE = {var: (pre2.pivot_table(index='Date', columns='Location', values=var, dropna=False)
             .reindex(index=full_dates, columns=stations))
        for var in ['Rainfall', 'Pressure3pm', 'Humidity3pm', '_RainToday']}
SPEC = [('Rainfall', False), ('Rainfall', True), ('Pressure3pm', False),
        ('Pressure3pm', True), ('Humidity3pm', False), ('_RainToday', False)]

def build_spatial(neighbours):
    """Bloc de features voisines pour un graphe donné, réaligné sur X_all."""
    out = meta[['Date', 'Location']].copy()
    cols = []
    for var, lag in SPEC:
        wide = WIDE[var]
        agg = pd.DataFrame({s: wide[neighbours[s]].mean(axis=1) for s in stations}, index=wide.index)
        if lag:
            agg = agg.shift(1)
        agg.index.name = 'Date'
        name = f'nbr_{var.replace("_RainToday","RainToday")}_{"t1" if lag else "t"}'
        out = out.merge(agg.reset_index().melt(id_vars='Date', var_name='Location', value_name=name),
                        on=['Date', 'Location'], how='left')
        cols.append(name)
    out['nearest_dist'] = out['Location'].map(NEAREST_DIST); cols.append('nearest_dist')
    return align_to_pipeline(out, cols), cols
print('builder ready')


## 5. Helpers + masques de split

Le masque chronologique n'est plus recalculé : ce sont les `N_TRAIN` premières lignes de `X_all`,
c'est-à-dire exactement `weatherAUS_X_train.csv`.


In [ ]:
def split_masks(split, X):
    """(masque train, CV, groupes) pour un protocole donné."""
    if split == 'chronological':
        tr = np.zeros(len(X), bool); tr[:N_TRAIN] = True
        return tr, TimeSeriesSplit(5), None
    gss = GroupShuffleSplit(1, test_size=0.2, random_state=RANDOM_STATE)
    a, _ = next(gss.split(X, y_all, groups=loc_all))
    tr = np.zeros(len(X), bool); tr[a] = True
    return tr, GroupKFold(5), loc_all[tr].reset_index(drop=True)

def light_eval(Xtr, ytr, Xte, yte, params=None):
    m = make_xgb(**(params or V2_PARAMS), scale_pos_weight=(1-ytr.mean())/ytr.mean())
    m.fit(Xtr, ytr); proba = m.predict_proba(Xte)[:, 1]
    p, r, _ = precision_recall_curve(yte, proba)
    return round(roc_auc_score(yte, proba), 4), round(auc(r, p), 4)

def pr_crossover_threshold(params, Xtr, ytr, cv, groups, grid=np.arange(0.10, 0.81, 0.01)):
    recs = []
    for tr, va in cv.split(Xtr, ytr, groups):
        m = make_xgb(**params); m.fit(Xtr.iloc[tr], ytr.iloc[tr])
        recs.append((ytr.iloc[va].values, m.predict_proba(Xtr.iloc[va])[:, 1]))
    best_t, best_d = 0.5, 1e9
    for t in grid:
        P = np.mean([precision_score(y, p >= t, zero_division=0) for y, p in recs])
        R = np.mean([recall_score(y, p >= t) for y, p in recs])
        F = np.mean([f1_score(y, p >= t) for y, p in recs])
        d = abs(P - F) + abs(F - R)
        if d < best_d: best_t, best_d = float(t), d
    return round(best_t, 2)

def full_eval(X, cols, params, split):
    tr, cv, groups = split_masks(split, X)
    Xtr, Xte = X.loc[tr, cols].reset_index(drop=True), X.loc[~tr, cols].reset_index(drop=True)
    ytr, yte = y_all[tr].reset_index(drop=True), y_all[~tr].reset_index(drop=True)
    pr = dict(params); pr['scale_pos_weight'] = (1 - ytr.mean()) / ytr.mean()
    t = pr_crossover_threshold(pr, Xtr, ytr, cv, groups)
    m = make_xgb(**pr); m.fit(Xtr, ytr); proba = m.predict_proba(Xte)[:, 1]; pred = (proba >= t).astype(int)
    p, r, _ = precision_recall_curve(yte, proba)
    return dict(split=split, threshold=t, F1=round(f1_score(yte, pred), 4),
                precision=round(precision_score(yte, pred), 4), recall=round(recall_score(yte, pred), 4),
                ROC_AUC=round(roc_auc_score(yte, proba), 4), PR_AUC=round(auc(r, p), 4)), m
print('helpers ready')


## 6. K sweep + directional (chronological, frozen config, ROC/PR)

In [ ]:
log_section('6. Spatial design sweep (chronological)')
rows = []
SPATIAL_COLS = {}
chrono = np.zeros(len(X_all), bool); chrono[:N_TRAIN] = True
for gname, graph in GRAPHS.items():
    sblock, scols = build_spatial(graph)
    Xg = pd.concat([X_all, sblock], axis=1)
    cols = BASE_FEATURES + TEMPORAL_FEATURES + scols
    roc, pr = light_eval(Xg.loc[chrono, cols].reset_index(drop=True), y_all[chrono].reset_index(drop=True),
                         Xg.loc[~chrono, cols].reset_index(drop=True), y_all[~chrono].reset_index(drop=True))
    rows.append({'graph': gname, 'n_nbr': len(next(iter(graph.values()))), 'ROC_AUC': roc, 'PR_AUC': pr})
    SPATIAL_COLS[gname] = scols
    print(f'  {gname:10s} -> ROC {roc:.4f}  PR {pr:.4f}')
sweep = pd.DataFrame(rows)
print('\n' + sweep.to_string(index=False))
best_graph = sweep.loc[sweep['PR_AUC'].idxmax(), 'graph']
print(f'\nBest spatial graph by PR-AUC: {best_graph}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(sweep['graph'], sweep['PR_AUC'], color=['steelblue' if gname != best_graph else 'crimson' for gname in sweep['graph']])
ax.set_ylabel('chronological PR-AUC'); ax.set_title('Spatial design sweep (frozen config)')
ax.set_ylim(sweep['PR_AUC'].min()-0.005, sweep['PR_AUC'].max()+0.005)
for i, v in enumerate(sweep['PR_AUC']): ax.text(i, v+0.0005, f'{v:.3f}', ha='center', fontsize=8)
plt.tight_layout(); save_fig('sptune_01_design_sweep', fig); plt.show()


## 7. Fresh hyperparameter search on the winning 44-feature set

In [ ]:
log_section('7. Fresh RandomizedSearchCV (winning spatial graph)')
sblock, scols = build_spatial(GRAPHS[best_graph])
X_win = pd.concat([X_all, sblock], axis=1)
WIN_COLS = BASE_FEATURES + TEMPORAL_FEATURES + scols

Xtr_w = X_win.loc[chrono, WIN_COLS].reset_index(drop=True)
ytr_w = y_all[chrono].reset_index(drop=True)
SPW = (1 - ytr_w.mean()) / ytr_w.mean()
print(f'Winning graph {best_graph} | {len(WIN_COLS)} features | train {len(Xtr_w):,}')

param_dist = {
    'n_estimators': [400, 600, 800, 1200], 'max_depth': [4, 5, 6, 8],
    'learning_rate': [0.02, 0.03, 0.05], 'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.9], 'min_child_weight': [3, 5, 10, 20],
    'gamma': [0, 0.5, 1.0, 2.0], 'reg_lambda': [1, 5, 10, 20], 'reg_alpha': [0, 0.5, 1.0],
    'scale_pos_weight': [SPW],
}
search = RandomizedSearchCV(make_xgb(), param_dist, n_iter=40, scoring='f1',
                            cv=TimeSeriesSplit(5), n_jobs=SEARCH_N_JOBS,
                            random_state=RANDOM_STATE, verbose=1)
search.fit(Xtr_w, ytr_w)
TUNED = {k: v for k, v in search.best_params_.items()}
print(f'Best params: {search.best_params_}')
print(f'Best CV F1 : {search.best_score_:.4f}')
save_model(search.best_estimator_, 'EXP-007_01_fresh_search',
           features=WIN_COLS, stage=f'RandomizedSearchCV sur le graphe {best_graph}',
           best_params=TUNED, cv_f1=float(search.best_score_), spatial_graph=best_graph)


## 8. Final comparison — frozen vs tuned config, both splits

In [ ]:
log_section('8. Final — frozen vs tuned (winning spatial graph)')
final_rows = []
frozen = {k: v for k, v in V2_PARAMS.items()}
for label, params in [('frozen (EXP-002 cfg)', frozen), ('tuned (fresh search)', TUNED)]:
    for split in ['chronological', 'leave-stations-out']:
        r, _m = full_eval(X_win, WIN_COLS, params, split)
        r = {'config': label, **r}; final_rows.append(r)
        print(f"  {label:22s} {r['split']:20s} -> F1 {r['F1']:.4f}  ROC {r['ROC_AUC']:.4f}  PR {r['PR_AUC']:.4f}")
        tag = 'frozen' if label.startswith('frozen') else 'tuned'
        save_model(_m, f"EXP-007_02_{tag}__{r['split'].replace('-', '_')}",
                   features=WIN_COLS, split=r['split'], stage=f'{label} / {r["split"]}',
                   params=params, threshold=r['threshold'], metrics=r, spatial_graph=best_graph)
final = pd.DataFrame(final_rows)
print('\n' + final.to_string(index=False))


## 9. Verdict

In [ ]:
log_section('9. Verdict')
fz = final[(final.config.str.startswith("frozen")) & (final.split == 'chronological')].iloc[0]
tn = final[(final.config.str.startswith("tuned")) & (final.split == 'chronological')].iloc[0]
print(f'Best spatial graph this run: {best_graph}')
print(f'Frozen cfg  chronological : F1 {fz.F1:.4f} / ROC {fz.ROC_AUC:.4f} / PR {fz.PR_AUC:.4f}')
print(f'Tuned  cfg  chronological : F1 {tn.F1:.4f} / ROC {tn.ROC_AUC:.4f} / PR {tn.PR_AUC:.4f}')
print(f'  fresh-search gain: dF1 {tn.F1-fz.F1:+.4f} | dROC {tn.ROC_AUC-fz.ROC_AUC:+.4f} | dPR {tn.PR_AUC-fz.PR_AUC:+.4f}')
if (tn.PR_AUC - fz.PR_AUC >= 0.003) or (tn.F1 - fz.F1 >= 0.003):
    print('\nVERDICT: la recherche fraîche améliore la config gelée. Adopter la nouvelle config.')
else:
    print('\nVERDICT: la config gelée (EXP-002) était déjà quasi optimale ; le tuning apporte peu.')
print(f'  -> graphe spatial retenu : {best_graph}')


## 10. Finalization

In [ ]:
log_section('Finalization')
print(f'Results file: {_PATH}')
figs = sorted(f for f in os.listdir('figures') if f.startswith('sptune_'))
for f in figs: print(f'  figures/{f}')
models = sorted(f for f in os.listdir(MODEL_DIR) if f.startswith('EXP-007_'))
print(f'Models saved ({len(models)}) in {MODEL_DIR}:')
for f in models: print(f'  {f}')
sys.stdout = _ORIG; _FILE.close()
ZIP = 'XGB_sptune_artifacts.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(_PATH)
    for f in figs: zf.write(f'figures/{f}')
print(f'Bundled -> {ZIP}')
if IN_COLAB:
    from google.colab import files; files.download(ZIP)
print(f'Done. See {_PATH} and figures/.')